# **Setup**


In [ ]:
# USE THIS CODE IF YOU ARE USING GOOGLE COLABORATORY
# this section connects the notebook to your google drive, allowing you to
# access files from and upload files to your google drive
from google.colab import drive
drive.mount('/content/drive')

# set current directory
# this should be the Google Drive folder where your file(s) are located
%cd "/content/drive/MyDrive/18.S997 Project/Data"


## verify current directory
!ls "/content/drive/MyDrive/18.S997 Project/Data"


# choose where you want your project files to be saved
project_folder = "/content/drive/MyDrive/18.S997 Project/Data"



Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1T3pm7UpOB513AURTmj_1_fLMQe3AO137/18.S997 Project/Data
 7k_onehot_and_smiles.csv		 dataset_all_cleaned.csv
 A_blissind_abxinfo_DropArray.xlsx	 Otava_abx_conc_162.csv
 all_data_clean.csv			 Otava_morgan_tsne.csv
 all_data.csv				 Otava_onehot_strain_abx.csv
 all_synergy_features.csv		 Otava_only.csv
 all_synergy_labeled.csv		'README Data.gdoc'
 Data_representation.gdoc		 v2_all_data_embedded.csv
 dataset_1_cleaned.csv			 xgboost_bliss_regression_model.json
 dataset_1_cleaned_from_duplicates.csv	 xgboost_feature_importance.png
 dataset_1.csv				 xgboost_pred_vs_actual.png


In [ ]:

!pip install rdkit
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors,Crippen
import numpy as np

from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split


import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader



from sklearn.ensemble import RandomForestRegressor

from torch.utils.data import Dataset, DataLoader

!pip install torch_geometric
import torch_geometric
from tqdm import tqdm

import numpy as np



# **Data import and processing**

In [ ]:
project_folder = "/content/drive/MyDrive/18.S997 Project/Data"
# Build full path to the file
file_name_dubl = "A_blissind_abxinfo_DropArray.xlsx"
file_path = os.path.join(project_folder, file_name_dubl)

# Load the Excel file
df = pd.read_excel("/content/drive/MyDrive/18.S997 Project/Data/A_blissind_abxinfo_DropArray.xlsx")

print(df.shape)
print(df.isnull().sum())
# Check the number of rows before dropping
print(f"Original dataset shape: {df.shape}")

# Drop all rows with any missing values
df_clean = df.dropna()

# Check the number of rows after dropping
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Removed {df.shape[0] - df_clean.shape[0]} rows with missing values")

# Verify no missing values remain
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

# Save the cleaned dataset
df_clean.to_csv("all_data_clean.csv", index=False)


In [ ]:
import pandas as pd
import numpy as np


# 1.  Build the label column
conditions = [
    df["bliss_med"] >= 0.30,   # synergistic
    df["bliss_med"] <= -0.30   # antagonistic
]
choices = [1, 0]

df["synergy_label"] = np.select(conditions, choices, default=np.nan)

# 2.  Drop the rows that fell into the “gray zone”
df = df.dropna(subset=["synergy_label"]).reset_index(drop=True)

# 3.  Optionally convert label to int for cleaner downstream use
df["synergy_label"] = df["synergy_label"].astype(int)

# df now contains only rows with synergy_label 1 or 0
# 4.  Save the cleaned‑up dataframe to disk
df.to_csv("all_synergy_labeled.csv", index=False)

v2_all_data_embedded creation: Includes finerprint embedding for abx and cpd for 1.3 million sized dataset

In [ ]:
# Build full path to the file
file_name_dubl = "A_blissind_abxinfo_DropArray.xlsx"
file_path = os.path.join(project_folder, file_name_dubl)

# Install openpyxl to read .xlsx files
!pip install openpyxl

# Load the Excel file
df_all = pd.read_excel(file_path)
df_all["abx_SMILES"] = df_all["abx_name"].map(abx_smiles_map)


# Helper to turn SMILES into fingerprint
def smiles_to_ecfp(smiles: str, n_bits: int = 1024, radius: int = 2) -> np.ndarray:
    vec = np.zeros((n_bits,), dtype=int)
    if pd.isna(smiles):
        return vec
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return vec
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    DataStructs.ConvertToNumpyArray(fp, vec)
    return vec

# Generate fingerprint matrices
abx_fp = np.vstack(df["abx_SMILES"].apply(smiles_to_ecfp))
cpd_fp = np.vstack(df["cp_SMILES"].apply(smiles_to_ecfp))

# Turn them into DataFrames with column names
abx_df = pd.DataFrame(abx_fp, columns=[f"abx_fp_{i}" for i in range(abx_fp.shape[1])])
cpd_df = pd.DataFrame(cpd_fp, columns=[f"cpd_fp_{i}" for i in range(cpd_fp.shape[1])])

# Numeric features
numeric_cols = ["abx_conc", "Ea_med", "Ec_med"]
numeric_df = df[numeric_cols].astype(float)

# One-hot encode strain
enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
strain_1hot = enc.fit_transform(df[["strain_name"]])
strain_df = pd.DataFrame(strain_1hot, columns=enc.get_feature_names_out(["strain_name"]))

# Concatenate all into a final DataFrame
X = pd.concat([abx_df, cpd_df, numeric_df, strain_df], axis=1)

# Target
y = df["bliss_med"]


full_df = pd.DataFrame(X)
full_df["target"] = y.values
full_df.to_csv("v2_all_data_embedded.csv", index=False)



# **XGBoost Regression Model**

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt


print("Loading dataset...")
full_df = pd.read_csv("v2_all_data_embedded.csv")
print(f"Dataset shape: {full_df.shape}")

# check for NaN, inf values in target before proceeding
print("Number of NaN values in target:", full_df["target"].isna().sum())
print("Number of infinite values in target:", np.isinf(full_df["target"]).sum())
print("Target value range:", full_df["target"].min(), "to", full_df["target"].max())


full_df = full_df.dropna(subset=["target"])  # remove rows with NaN targets
full_df = full_df.replace([np.inf, -np.inf], np.nan).dropna()

# features and target cleaning
X = full_df.drop(columns=["target"])
y = full_df["target"]
print(f"Features shape after cleaning: {X.shape}")
print(f"Target shape after cleaning: {y.shape}")

# split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training data: {X_train.shape}")
print(f"Testing data: {X_test.shape}")

# XGBoost regression model, Kfold split
rmse_scores = []
r2_scores = []
mae_scores = []

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

X_np = X.values.astype(np.float32)
y_np = y.values.astype(np.float32)

# check invalid
assert not np.isnan(y_np).any(), "NaN values still present in target"
assert not np.isinf(y_np).any(), "Infinite values still present in target"

for fold, (train_idx, test_idx) in enumerate(kf.split(X_np)):
    print(f"\n=== Fold {fold+1}/{n_splits} ===")

    X_train, X_test = X_np[train_idx], X_np[test_idx]
    y_train, y_test = y_np[train_idx], y_np[test_idx]

    assert not np.isnan(y_train).any(), f"NaN values in y_train fold {fold+1}"
    assert not np.isinf(y_train).any(), f"Inf values in y_train fold {fold+1}"

    # model
    xgb_model = xgb.XGBRegressor(
        reg_alpha=0.1,
        reg_lambda=0.1,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9
    )

    xgb_model.fit(X_train, y_train)

    # predictions
    y_pred = xgb_model.predict(X_test)

    # metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    rmse_scores.append(rmse)
    r2_scores.append(r2)
    mae_scores.append(mae)

    print(f"Fold {fold+1} RMSE: {rmse:.4f}")
    print(f"Fold {fold+1} R²: {r2:.4f}")
    print(f"Fold {fold+1} MAE: {mae:.4f}")

print(f"\n=== Cross-Validation Summary ===")
print(f"Mean RMSE: {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"Mean R²: {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
print(f"Mean MAE: {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")

print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")

# get pearson R with numpy arrays
pearson_r = np.corrcoef(y_test, y_pred)[0, 1]
print(f"Pearson R: {pearson_r:.4f}")

# feature importance
print("\n=== Feature Importance ===")
importance = xgb_model.feature_importances_
feature_names = X.columns.tolist()
feature_importance = sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)
for feature, importance in feature_importance[:20]:  # show top 20 features
    print(f"{feature}: {importance:.4f}")

xgb_model.save_model('xgboost_bliss_regression_model.json')
print("\nModel saved as 'xgboost_bliss_regression_model.json'")


## **XGBoost Regression Plots**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

cutoff_coloring = True

import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

# Set rcParams BEFORE figure
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 14

width_cm = 12
height_cm = 10

plt.figure(figsize=(width_cm/2.54, height_cm/2.54))
ax = sns.scatterplot(
    x=y_test,
    y=y_pred,
    c='black',
    edgecolors='black',
    alpha=0.5,
    s=15,
    linewidths=0.3,
    rasterized=True
    )

pearson_r = np.corrcoef(y_test, y_pred)[0, 1]
r2 = r2_score(y_test, y_pred)
# Pearson r in a box
plt.text(
    0.95, 0.2, f'Pearson R = {pearson_r:.2f}\n R2 = {r2:.2f}',
    transform=plt.gca().transAxes,
    fontsize=12, verticalalignment='top', horizontalalignment='right',
    bbox=dict(boxstyle='round', facecolor='white', edgecolor='black')
)

# Axis labels and title
plt.xlabel('Actual Bliss Score')
plt.ylabel('Predicted Bliss Score')
#plt.title(f'XGBoost Regression Model Performance')

# Ticks formatting (this ensures numeric axis labels show up and are styled)
ax.tick_params(
    axis='both', which='both',
    direction='out', length=5, width=1, labelsize=11,
    bottom=True, top=False, left=True, right=False
)

# Remove box
sns.despine()

# Tight layout
plt.tight_layout()
plt.grid(False)

plt.savefig(
    f'./Regression_correlation.pdf',
    dpi=300, bbox_inches='tight'
    )
plt.show()

###################################################################

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Set style
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 14

# Figure size
width_cm = 12
height_cm = 10

# Define threshold and colors
threshold = 0.3
highlight_color = '#CD726E'
default_color = 'black'

# Assign color conditionally
full_df['color'] = full_df['target'].apply(lambda x: highlight_color if x > threshold else default_color)

# Plot
plt.figure(figsize=(width_cm/2.54, height_cm/2.54))
ax = plt.gca()

# Use the 'color' column directly
plt.scatter(
    full_df['Ea_med'],
    full_df['target'],
    c=full_df['color'],
    edgecolors='black',
    alpha=0.5,
    s=15,
    linewidths=0.3,
    rasterized=True
)

# Add horizontal dotted line at threshold
plt.axhline(y=threshold, color='grey', linestyle='dotted', linewidth=1)

# Pearson correlation
pearson_r = full_df['target'].corr(full_df['Ea_med'])
plt.text(
    0.95, 0.95, f'Pearson R = {pearson_r:.2f}',
    transform=ax.transAxes,
    fontsize=12, verticalalignment='top', horizontalalignment='right',
    bbox=dict(boxstyle='round', facecolor='white', edgecolor='black')
)

# Labels
plt.xlabel('Ea Score')
plt.ylabel('Bliss Score')

# Formatting
ax.tick_params(
    axis='both', which='both',
    direction='out', length=5, width=1, labelsize=11,
    bottom=True, top=False, left=True, right=False
)
sns.despine()
plt.tight_layout()
plt.grid(False)

# Save and show
plt.savefig('./Ea_vs_Bliss_correlation.pdf', dpi=300, bbox_inches='tight')
plt.show()



# **Antibiotics to SMILES**

In [ ]:
# returns a NumPy array of the distinct antibiotic names
unique_abx = df["abx_name"].unique()

# …or, if you prefer a plain Python list (and sorted alphabetically)
unique_abx_list = sorted(df["abx_name"].unique())

print(unique_abx_list)

abx_smiles_map = {
    # aminoglycoside
    "AMK":  "C1C(C(C(C(C1NC(=O)C(CCN)O)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(C(C(O3)CN)O)O)O)N",  # amikacin :contentReference[oaicite:0]{index=0}

    # β‑lactams
    "AMP":  "CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O)O)C",                          # ampicillin :contentReference[oaicite:1]{index=1}
    "CEP":  "CC(=O)OCC1=C(N2C(C(C2=O)NC(=O)CC3=CC=CS3)SC1)C(=O)O",                         # cephalothin :contentReference[oaicite:2]{index=2}
    "CEX":  "CC1=C(N2C(C(C2=O)NC(=O)C(C3=CC=CC=C3)N)SC1)C(=O)O",                          # cephalexin :contentReference[oaicite:3]{index=3}
    "IMP":  "CC(C1C2CC(=C(N2C1=O)C(=O)O)SCCN=CN)O",                                       # imipenem :contentReference[oaicite:4]{index=4}

    # macrolides / lincosamide
    "AZI":  "CCC1C(C(C(C(=O)C(CC(C(C(C(C(=O)O1)C)OC2CC(C(C(O2)C)O)(C)OC)C)OC3C(C(CC(O3)C)N(C)C)O)(C)O)C)C)O)(C)O",  # azithromycin :contentReference[oaicite:5]{index=5}
    "ERY":  "CCC1C(C(C(C(=O)C(CC(C(C(C(C(=O)O1)C)OC2CC(C(C(O2)C)O)(C)OC)C)OC3C(C(CC(O3)C)N(C)C)O)(C)O)C)C)O)(C)O",  # erythromycin :contentReference[oaicite:6]{index=6}
    "CLIN": "CCCC1CC(N(C1)C)C(=O)NC(C2C(C(C(C(O2)SC)O)O)O)C(C)Cl",                        # clindamycin :contentReference[oaicite:7]{index=7}

    # peptide / lipopeptide antibiotics
    "BAC":  "CCC(C)C(N)NC4=NC(C(=O)N[C@@H](CC(C)C)C(=O)N[C@H](CCC(O)=O)C(=O)N[C@@H](C(C)CC)C(=O)N[C@H]3CCCCNC(=O)[C@H](CC(N)=O)NC(=O)[C@@H](CC(O)=O)NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@@H](Cc2ccccc2)NC(=O)[C@H](C(C)CC)NC(=O)[C@@H](CCCN)NC3=O)CS4",  # bacitracin A :contentReference[oaicite:8]{index=8}
    "COLIS":"CCC(C)CCCC(=O)NC(CCN)C(=O)NC(C(C)O)C(=O)NC(CCN)C(=O)NC1CCNC(=O)…",          # colistin (truncated for brevity) :contentReference[oaicite:9]{index=9}
    "DAP":  "CCNC(=O)C1CCCN1C(=O)C(CCCNC(=N)N)NC(=O)C(CC(C)C)NC(=O)C(CC2=CNC3=CC=CC=C32)NC(=O)C(CC4=CC=C(C=C4)O)NC(=O)C(CO)NC(=O)C(CC5=CNC6=CC=CC=C65)NC(=O)C(CC7=CN=CN7)NC(=O)C8CCC(=O)N8",  # daptomycin :contentReference[oaicite:10]{index=10}
    "VANC": "CC1C(C(CC(O1)OC2C(C(C(OC2OC3=C4C=C5C=C3OC6=C(C=C(C=C6)C(C(C(=O)NC(C(=O)NC5C(=O)NC7C8=CC(=C(C=C8)O)C9=C(C=C(C=C9O)O)C(NC(=O)C(C(C1=CC(=C(O4)C=C1)Cl)O)NC7=O)C(=O)O)CC(=O)N)NC(=O)C(CC(C)C)NC)O)Cl)CO)O)O)(C)N)O",  # vancomycin :contentReference[oaicite:11]{index=11}

    # fluoroquinolones
    "LEVO": "CC1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)F)C(=O)O",                        # levofloxacin :contentReference[oaicite:12]{index=12}
    "MOX":  "COC1=C2C(=CC(=C1N3CC4CCCNC4C3)F)C(=O)C(=CN2C5CC5)C(=O)O",                  # moxifloxacin :contentReference[oaicite:13]{index=13}

    # rifamycin
    "RIF":  "CC1C=CC=C(C(=O)NC2=C(C3=C(C4=C(C(=C3O)C)OC(C4=O)(OC=CC(C(C(C(C(C(C1O)C)O)C)OC(=O)C)C)OC)C)C(=O)C2=CNN5CCN(CC5)C)O)C",  # rifampicin :contentReference[oaicite:14]{index=14}

    # oxazolidinone
    "LIN":  "CC(=O)NCC1CN(C(=O)O1)C2=CC(=C(C=C2)N3CCOCC3)F",                            # linezolid :contentReference[oaicite:15]{index=15}

    # nitroimidazole
    "MET":  "CC1=NC=C(N1CCO)[N+](=O)[O-]",                                             # metronidazole :contentReference[oaicite:16]{index=16}

    # phosphonic acid
    "FOS":  "CC1C(O1)P(=O)(O)O",                                                       # fosfomycin :contentReference[oaicite:17]{index=17}

    # sulfonamides
    "SFA":  "CC(=O)NS(=O)(=O)C1=CC=C(C=C1)N",                                           # sulfacetamide :contentReference[oaicite:18]{index=18}
    "SUL":  "CC1=CC(=NO1)NS(=O)(=O)C2=CC=C(C=C2)N",                                     # sulfamethoxazole :contentReference[oaicite:19]{index=19}

    # diaminopyrimidine
    "TRIM":"COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N",                                     # trimethoprim :contentReference[oaicite:20]{index=20}

    # tetracyclines
    "TET":  "CC1(C2CC3C(C(=O)C(=C(C3(C(=O)C2=C(C4=C1C=CC=C4O)O)O)O)C(=O)N)N(C)C)O",     # tetracycline :contentReference[oaicite:21]{index=21}
    "TIGE": "CC(C)(C)NCC(=O)NC1=CC(=C2CC3CC4C(C(=O)C(=C(C4(C(=O)C3=C(C2=C1O)O)O)O)C(=O)N)N(C)C)N(C)C",  # tigecycline :contentReference[oaicite:22]{index=22}

    # glycopeptide / others
    "NOV":  "CC1=C(C=CC2=C1OC(=O)C(=C2O)NC(=O)C3=CC(=C(C=C3)O)CC=C(C)C)OC4C(C(C(C(O4)(C)C)OC)OC(=O)N)O",  # novobiocin :contentReference[oaicite:23]{index=23}
    "CYC":  "C1C(C(=O)NO1)N",                                                           # cycloserine :contentReference[oaicite:24]{index=24}
}



# **7k dataset trained classifiers**

## **ChemBERTa Embedding Models (for 7k dataset)**

### ChemBERTa Model Import

In [ ]:

import json
from functools import lru_cache
from typing import Tuple

import numpy as np
import torch
from transformers import AutoTokenizer, RobertaForSequenceClassification



CHECKPOINTS = {
    # Hugging Face model hub IDs
    "ChemBERTa-77M-MTR": "DeepChem/ChemBERTa-77M-MTR",
    "MolFormer": "ibm/molformer-768-4m",                # rotary linear‑attention
}


# Model loading


@lru_cache(maxsize=None)
def load_model(which: str = "ChemBERTa-77M-MTR") -> Tuple[AutoTokenizer, RobertaForSequenceClassification]:
    """
    Load tokenizer + model, move model to the best available device,
    set to eval() and return the pair. Caches the result so subsequent
    calls are instant.
    """
    if which not in CHECKPOINTS:
        raise ValueError(f"`which` must be one of {list(CHECKPOINTS)}")

    ckpt = CHECKPOINTS[which]
    print(f"[embed_smiles] Loading checkpoint: {ckpt}", flush=True)

    # Pick GPU if available; fall back to CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    model = RobertaForSequenceClassification.from_pretrained(ckpt)
    model.to(device).eval()  # inference‑only mode

    # Freeze parameters
    for p in model.parameters():
        p.requires_grad_(False)

    return tokenizer, model


# Embedding extraction


@torch.no_grad()
def smiles_to_vec(smiles: str,
                  tokenizer: AutoTokenizer,
                  model: RobertaForSequenceClassification) -> np.ndarray:
    """
    Convert a single SMILES string to a 768‑dimensional NumPy vector
    using the model's CLS token embedding.

    Parameters
    ----------
    smiles : str
        Canonical or raw SMILES.
    tokenizer, model
        Outputs of `load_model`.

    Returns
    -------
    np.ndarray, shape = (768,)
    """
    toks = tokenizer(smiles,
                     return_tensors="pt",
                     padding=False,
                     truncation=False)
    toks = {k: v.to(model.device) for k, v in toks.items()}

    # Forward pass: obtain hidden states
    hidden = model(**toks).logits
    cls = hidden.squeeze(0)

    return cls.cpu().numpy()





def run_smiles_embedding(smiles: str, model_name: str = "ChemBERTa-77M-MTR"):
    tok, lm = load_model(model_name)
    vec = smiles_to_vec(smiles, tok, lm)
    return vec




In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from sklearn.preprocessing import OneHotEncoder

# Load the data
df = pd.read_csv("all_synergy_labeled.csv")

# Map abx SMILES
df["abx_SMILES"] = df["abx_name"].map(abx_smiles_map)


# Numeric features
numeric_cols = ["abx_conc", "Ea_med", "Ec_med"]
numeric_df = df[numeric_cols].astype(float)

# One-hot encode strain
enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
strain_1hot = enc.fit_transform(df[["strain_name"]])
strain_df = pd.DataFrame(strain_1hot, columns=enc.get_feature_names_out(["strain_name"]))

# Concatenate all into a final DataFrame
X = pd.concat([numeric_df, strain_df], axis=1)

# Target
y = df["synergy_label"].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print(X.head())


In [ ]:
#distinct antibiotic names
unique_abx = df["abx_name"].unique()
unique_abx_list = sorted(df["abx_name"].unique())

In [ ]:
# Combine X (features) and y (target)
combined_df = pd.concat([X, y.rename("synergy_label")], axis=1)
columns_to_add = ["abx_SMILES", "cp_SMILES"]
combined_df = pd.concat([combined_df, df[columns_to_add]], axis=1)


print(combined_df.head())
combined_df.to_csv("7k_onehot_and_smiles.csv", index=False)


In [ ]:
df_wo_emb = pd.read_csv("7k_onehot_and_smiles.csv")
print(df_wo_emb.head())

In [ ]:

def get_smiles_embedding(smiles: str, model_name: str = "ChemBERTa-77M-MTR"):
    tok, lm = load_model(model_name)
    vec = smiles_to_vec(smiles, tok, lm)
    return vec

# Apply the embedding extraction to each row using tqdm for progress tracking
df_wo_emb["abx_embedding"] = [
    get_smiles_embedding(str(smiles)) for smiles in tqdm(df_wo_emb["abx_SMILES"])
]

df_wo_emb["cp_embedding"] = [
    get_smiles_embedding(str(smiles)) for smiles in tqdm(df_wo_emb["cp_SMILES"])
]

# Display the updated DataFrame
df_wo_emb.head()

In [ ]:
invalid_rows = []

for idx, (abx_emb, cp_emb) in enumerate(zip(df_wo_emb["abx_embedding"], df_wo_emb["cp_embedding"])):
    if not (isinstance(abx_emb, np.ndarray) and abx_emb.shape == (199,)):
        print(f"Row {idx} - Invalid abx_embedding: {type(abx_emb)}, {abx_emb.shape if isinstance(abx_emb, np.ndarray) else 'Not an array'}")
        invalid_rows.append(idx)
    if not (isinstance(cp_emb, np.ndarray) and cp_emb.shape == (199,)):
        print(f"Row {idx} - Invalid cp_embedding: {type(cp_emb)}, {cp_emb.shape if isinstance(cp_emb, np.ndarray) else 'Not an array'}")
        invalid_rows.append(idx)

In [ ]:

X = pd.DataFrame(
    df_wo_emb.apply(
        lambda row: np.concatenate([row["abx_embedding"], row["cp_embedding"]]), axis=1
    ).tolist()
)

# Assigning column names for better understanding
X.columns = [f"feature_{i}" for i in range(X.shape[1])]

# Adding the original columns (abx_conc and stain) if you want them in the feature set
X["abx_conc"] = df_wo_emb["abx_conc"].values
X["strain_name_Ab17978"] = df_wo_emb["strain_name_Ab17978"].values
X["strain_name_AbLac4"] = df_wo_emb["strain_name_AbLac4"].values
X["strain_name_Kp0087"] = df_wo_emb["strain_name_Kp0087"].values
X["strain_name_Kp43816"] =  df_wo_emb["strain_name_Kp43816"].values
X["strain_name_PAO1"] = df_wo_emb["strain_name_PAO1"].values
X["strain_name_Pa0095"] = df_wo_emb["strain_name_Pa0095"].values


# Example target variable (replace with your actual target)
y = df_wo_emb['synergy_label']  # Replace this with your actual target column



# Displaying the resulting X DataFrame
print(X.head())



## LightGBM Model

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
from sklearn.model_selection import KFold


# Create KFold object
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Store metrics for averaging
aucs = []
accuracies = []
conf_matrices = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    print(f"\n--- Fold {fold} ---")

    X_train, X_test = X.iloc[train_idx].astype(np.float32), X.iloc[test_idx].astype(np.float32)
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    clf = lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )

    try:
        clf.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, first_metric_only=True),
                lgb.log_evaluation(period=50),
            ],
        )
    except TypeError:
        clf.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="auc",
            early_stopping_rounds=50,
            verbose=False,
        )

    proba = clf.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, proba)
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred).ravel()  # TN, FP, FN, TP

    print(f"Best iteration : {clf.best_iteration_}")
    print(f"AUROC          : {auc:.4f}")
    print(f"Accuracy       : {acc:.4f}")
    print(f"Confusion matrix (TN, FP, FN, TP): {cm}")

    aucs.append(auc)
    accuracies.append(acc)
    conf_matrices.append(cm)

# Average metrics
avg_auc = np.mean(aucs)
avg_acc = np.mean(accuracies)
sum_cm = np.sum(conf_matrices, axis=0)

print("\n=== Cross-Validation Summary ===")
print(f"Average AUROC    : {avg_auc:.4f}")
print(f"Average Accuracy : {avg_acc:.4f}")
print(f"Total Confusion Matrix (sum over folds): {sum_cm}")


In [ ]:

# PLOTS for LightGBM model performance

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay
import numpy as np

#1) ROC curve
fpr, tpr, _ = roc_curve(y_test, proba)
roc_auc      = auc(fpr, tpr)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, lw=2, label=f"AUROC = {roc_auc:0.3f}")
plt.plot([0, 1], [0, 1], ls="--", lw=1)
plt.xlabel("False‑Positive Rate")
plt.ylabel("True‑Positive Rate")
plt.title("ROC curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# 2) Confusion‑matrix heatmap
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    cmap="Blues",
    colorbar=False,
)
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

# 3) Feature‑importance plot
# LightGBM returns importance values for every feature column (same order as X)
importances = clf.feature_importances_
#top 20
N = 20
idx_top     = np.argsort(importances)[-N:]
plt.figure(figsize=(6, 4 + 0.25 * N))
plt.barh(range(N), importances[idx_top], align="center")
plt.yticks(range(N), [f"feat_{i}" for i in idx_top])
plt.xlabel("Gain‑based importance")
plt.title(f"Top {N} features")
plt.tight_layout()
plt.show()


## XGBoost

In [ ]:
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import numpy as np
from sklearn.model_selection import KFold



aucs, accuracies = [], []
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n=== Fold {fold+1}/{n_splits} ===")

    # Use .iloc to select rows by integer location
    X_train, X_test = X.iloc[train_idx].astype(np.float32), X.iloc[test_idx].astype(np.float32)
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)

    pred = xgb_model.predict(X_test)
    proba = xgb_model.predict_proba(X_test)[:, 1]
    auc_score = auc(*roc_curve(y_test, proba)[:2])

    aucs.append(auc_score)
    accuracies.append(xgb_model.score(X_test, y_test))

    print(f"Fold {fold+1} AUROC   : {auc_score:.4f}")
    print(f"Fold {fold+1} Accuracy: {accuracies[-1]:.4f}")

# Cross-Validation Summary
print(f"\n=== Cross-Validation Summary ===")
print(f"Mean AUROC   : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"Mean Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")

# 1) ROC Curve Plot
fpr, tpr, _ = roc_curve(y_test, proba)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUROC = {auc_score:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# 2) Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix")
plt.show()

# 3) Feature Importance Plot
importances = xgb_model.feature_importances_
N = 20
idx_top = np.argsort(importances)[-N:]

plt.figure(figsize=(6, 4 + 0.25 * N))
plt.barh(range(N), importances[idx_top], align="center")
plt.yticks(range(N), [f"feat_{i}" for i in idx_top])
plt.xlabel("Gain-based Importance")
plt.title(f"Top {N} Features")
plt.tight_layout()
plt.show()

## Random Forest

In [ ]:
### RandomForestClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
from sklearn.model_selection import KFold
aucs, accuracies = [], []

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n=== Fold {fold+1}/{n_splits} ===")

    # Use .iloc for integer-location based indexing to select rows
    X_train, X_test = X.iloc[train_idx].astype(np.float32), X.iloc[test_idx].astype(np.float32)
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx] # Apply the same fix to y

    rf = RandomForestClassifier(n_estimators=200)
    rf.fit(X_train, y_train)
    acc = rf.score(X_test, y_test)
    pred = rf.predict(X_test)
    auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

    aucs.append(auc)
    accuracies.append(acc)

    print(f"Fold {fold+1} AUROC   : {auc:.4f}")
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")
    print("Confusion matrix (TN, FP, FN, TP):", confusion_matrix(y_test, pred).ravel())

print(f"\n=== Cross-Validation Summary ===")
print(f"Mean AUROC   : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"Mean Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")

## Logistic Regression

In [ ]:
#Logistic Regression

from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
from sklearn.model_selection import KFold
aucs, accuracies = [], []

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)



for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n=== Fold {fold+1}/{n_splits} ===")

    # Use .iloc for integer-location based indexing to select rows
    X_train, X_test = X.iloc[train_idx].astype(np.float32), X.iloc[test_idx].astype(np.float32)
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx] # Apply the same fix to y

    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    acc = lr.score(X_test, y_test)
    pred = lr.predict(X_test)
    auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])

    aucs.append(auc)
    accuracies.append(acc)

    print(f"Fold {fold+1} AUROC   : {auc:.4f}")
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")
    print("Confusion matrix (TN, FP, FN, TP):", confusion_matrix(y_test, pred).ravel())


print(f"\n=== Cross-Validation Summary ===")
print(f"Mean AUROC   : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"Mean Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")

## **MLP classifier**

### Data processing

In [ ]:
df = pd.read_csv("all_synergy_labeled.csv")
df.shape

In [ ]:

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from sklearn.preprocessing import OneHotEncoder

abx_smiles_map = {
    # aminoglycoside
    "AMK":  "C1C(C(C(C(C1NC(=O)C(CCN)O)OC2C(C(C(C(O2)CO)O)N)O)O)OC3C(C(C(C(O3)CN)O)O)O)N",  # amikacin :contentReference[oaicite:0]{index=0}

    # β‑lactams
    "AMP":  "CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O)O)C",                          # ampicillin :contentReference[oaicite:1]{index=1}
    "CEP":  "CC(=O)OCC1=C(N2C(C(C2=O)NC(=O)CC3=CC=CS3)SC1)C(=O)O",                         # cephalothin :contentReference[oaicite:2]{index=2}
    "CEX":  "CC1=C(N2C(C(C2=O)NC(=O)C(C3=CC=CC=C3)N)SC1)C(=O)O",                          # cephalexin :contentReference[oaicite:3]{index=3}
    "IMP":  "CC(C1C2CC(=C(N2C1=O)C(=O)O)SCCN=CN)O",                                       # imipenem :contentReference[oaicite:4]{index=4}

    # macrolides / lincosamide
    "AZI":  "CCC1C(C(C(C(=O)C(CC(C(C(C(C(=O)O1)C)OC2CC(C(C(O2)C)O)(C)OC)C)OC3C(C(CC(O3)C)N(C)C)O)(C)O)C)C)O)(C)O",  # azithromycin :contentReference[oaicite:5]{index=5}
    "ERY":  "CCC1C(C(C(C(=O)C(CC(C(C(C(C(=O)O1)C)OC2CC(C(C(O2)C)O)(C)OC)C)OC3C(C(CC(O3)C)N(C)C)O)(C)O)C)C)O)(C)O",  # erythromycin :contentReference[oaicite:6]{index=6}
    "CLIN": "CCCC1CC(N(C1)C)C(=O)NC(C2C(C(C(C(O2)SC)O)O)O)C(C)Cl",                        # clindamycin :contentReference[oaicite:7]{index=7}

    # peptide / lipopeptide antibiotics
    "BAC":  "CCC(C)C(N)NC4=NC(C(=O)N[C@@H](CC(C)C)C(=O)N[C@H](CCC(O)=O)C(=O)N[C@@H](C(C)CC)C(=O)N[C@H]3CCCCNC(=O)[C@H](CC(N)=O)NC(=O)[C@@H](CC(O)=O)NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@@H](Cc2ccccc2)NC(=O)[C@H](C(C)CC)NC(=O)[C@@H](CCCN)NC3=O)CS4",  # bacitracin A :contentReference[oaicite:8]{index=8}
    "COLIS":"CCC(C)CCCC(=O)NC(CCN)C(=O)NC(C(C)O)C(=O)NC(CCN)C(=O)NC1CCNC(=O)…",          # colistin (truncated for brevity) :contentReference[oaicite:9]{index=9}
    "DAP":  "CCNC(=O)C1CCCN1C(=O)C(CCCNC(=N)N)NC(=O)C(CC(C)C)NC(=O)C(CC2=CNC3=CC=CC=C32)NC(=O)C(CC4=CC=C(C=C4)O)NC(=O)C(CO)NC(=O)C(CC5=CNC6=CC=CC=C65)NC(=O)C(CC7=CN=CN7)NC(=O)C8CCC(=O)N8",  # daptomycin :contentReference[oaicite:10]{index=10}
    "VANC": "CC1C(C(CC(O1)OC2C(C(C(OC2OC3=C4C=C5C=C3OC6=C(C=C(C=C6)C(C(C(=O)NC(C(=O)NC5C(=O)NC7C8=CC(=C(C=C8)O)C9=C(C=C(C=C9O)O)C(NC(=O)C(C(C1=CC(=C(O4)C=C1)Cl)O)NC7=O)C(=O)O)CC(=O)N)NC(=O)C(CC(C)C)NC)O)Cl)CO)O)O)(C)N)O",  # vancomycin :contentReference[oaicite:11]{index=11}

    # fluoroquinolones
    "LEVO": "CC1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)F)C(=O)O",                        # levofloxacin :contentReference[oaicite:12]{index=12}
    "MOX":  "COC1=C2C(=CC(=C1N3CC4CCCNC4C3)F)C(=O)C(=CN2C5CC5)C(=O)O",                  # moxifloxacin :contentReference[oaicite:13]{index=13}

    # rifamycin
    "RIF":  "CC1C=CC=C(C(=O)NC2=C(C3=C(C4=C(C(=C3O)C)OC(C4=O)(OC=CC(C(C(C(C(C(C1O)C)O)C)OC(=O)C)C)OC)C)C(=O)C2=CNN5CCN(CC5)C)O)C",  # rifampicin :contentReference[oaicite:14]{index=14}

    # oxazolidinone
    "LIN":  "CC(=O)NCC1CN(C(=O)O1)C2=CC(=C(C=C2)N3CCOCC3)F",                            # linezolid :contentReference[oaicite:15]{index=15}

    # nitroimidazole
    "MET":  "CC1=NC=C(N1CCO)[N+](=O)[O-]",                                             # metronidazole :contentReference[oaicite:16]{index=16}

    # phosphonic acid
    "FOS":  "CC1C(O1)P(=O)(O)O",                                                       # fosfomycin :contentReference[oaicite:17]{index=17}

    # sulfonamides
    "SFA":  "CC(=O)NS(=O)(=O)C1=CC=C(C=C1)N",                                           # sulfacetamide :contentReference[oaicite:18]{index=18}
    "SUL":  "CC1=CC(=NO1)NS(=O)(=O)C2=CC=C(C=C2)N",                                     # sulfamethoxazole :contentReference[oaicite:19]{index=19}

    # diaminopyrimidine
    "TRIM":"COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N",                                     # trimethoprim :contentReference[oaicite:20]{index=20}

    # tetracyclines
    "TET":  "CC1(C2CC3C(C(=O)C(=C(C3(C(=O)C2=C(C4=C1C=CC=C4O)O)O)O)C(=O)N)N(C)C)O",     # tetracycline :contentReference[oaicite:21]{index=21}
    "TIGE": "CC(C)(C)NCC(=O)NC1=CC(=C2CC3CC4C(C(=O)C(=C(C4(C(=O)C3=C(C2=C1O)O)O)O)C(=O)N)N(C)C)N(C)C",  # tigecycline :contentReference[oaicite:22]{index=22}

    # glycopeptide / others
    "NOV":  "CC1=C(C=CC2=C1OC(=O)C(=C2O)NC(=O)C3=CC(=C(C=C3)O)CC=C(C)C)OC4C(C(C(C(O4)(C)C)OC)OC(=O)N)O",  # novobiocin :contentReference[oaicite:23]{index=23}
    "CYC":  "C1C(C(=O)NO1)N",                                                           # cycloserine :contentReference[oaicite:24]{index=24}
}


df = pd.read_csv("all_synergy_labeled.csv")


# map abx smiles
df["abx_SMILES"] = df["abx_name"].map(abx_smiles_map)


# Helper to turn a SMILES into a fingerprint

def smiles_to_ecfp(smiles: str, n_bits: int = 1024, radius: int = 2) -> np.ndarray:
    vec = np.zeros((n_bits,), dtype=int)
    if pd.isna(smiles):
        return vec
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return vec
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    DataStructs.ConvertToNumpyArray(fp, vec)
    return vec


# fingerprint matrices for antibiotic and compound

abx_fp   = np.vstack(df["abx_SMILES"].apply(smiles_to_ecfp))
cpd_fp   = np.vstack(df["cp_SMILES"].apply(smiles_to_ecfp))


# Numeric features: dose + Ea_med + Ec_med
numeric_cols = ["abx_conc"] #, "Ea_med", "Ec_med" , "Ea_med"
numeric_feats = df[numeric_cols].to_numpy(dtype=float)


# One‑hot encode the strain

enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
strain_1hot = enc.fit_transform(df[["strain_name"]])


# Concatenate every block → final feature matrix X
X = np.hstack([abx_fp, cpd_fp, numeric_feats, strain_1hot])

# Target vector y
y = df["synergy_label"].to_numpy(dtype=int)    # 0 = antagonistic, 1 = synergistic

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

class abxDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X_item = self.X[idx]
        y_item = self.y[idx]

        # Convert X to torch.float32 (features) and y to torch.long (labels)
        return torch.tensor(X_item, dtype=torch.float32), torch.tensor(y_item, dtype=torch.long)

  #split into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=1/8, random_state=42)





### Model Definition

In [ ]:
class cpClassifier(nn.Module):
    def __init__(self, hidden_width, hidden_depth, activation):
        super().__init__()

        self.hidden_width = hidden_width
        self.hidden_depth = hidden_depth
        self.activation = activation

        act_fn = nn.ReLU() if activation == 'relu' else nn.LeakyReLU()
        input_dim = X_train.shape[1]

        layers = []

        # Input layer
        layers.append(nn.Linear(input_dim, hidden_width))
        layers.append(nn.BatchNorm1d(hidden_width))
        layers.append(act_fn)
        layers.append(nn.Dropout(0.4))

        # Hidden layers
        for _ in range(hidden_depth - 1):
            layers.append(nn.Linear(hidden_width, hidden_width))
            layers.append(nn.BatchNorm1d(hidden_width))
            layers.append(act_fn)
            layers.append(nn.Dropout(0.4))

        # Output layer: one logit
        layers.append(nn.Linear(hidden_width, 1))

        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x).squeeze(1)  # [batch_size]


### Training and test loops

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

class EarlyStopping:
    def __init__(self, tolerance=5, min_delta=0.001):
        self.tolerance = tolerance
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, validation_loss):
        if self.best_loss is None:
            self.best_loss = validation_loss
        elif validation_loss < self.best_loss - self.min_delta:
            self.best_loss = validation_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.tolerance:
                self.early_stop = True

def train(model, dataloader, optimizer, device):
    '''
    Train the model for one epoch.
    '''
    batch_loss = []
    batch_correct = 0
    total_samples = 0

    model.train()  # Set model to training mode
    loss_function = nn.BCEWithLogitsLoss()

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)

        y = y.float()

        optimizer.zero_grad()
        outputs = model(X)  # logits: shape [batch_size, num_classes]
        loss = loss_function(outputs, y)  # y should be [batch_size] with class indices

        loss.backward()
        optimizer.step()

        batch_loss.append(loss.item())

        # Compute number of correct predictions
        preds = (torch.sigmoid(outputs) > 0.5).long()
        batch_correct += torch.sum(preds == y).item()
        total_samples += y.size(0)

    avg_loss = np.mean(batch_loss)
    accuracy = batch_correct / total_samples

    return avg_loss, accuracy


def validate(model, dataloader, device):
    '''
    Validate the model on validation dataset.
    '''
    val_loss = []
    val_correct = 0
    total_samples = 0

    model.eval()  # Set model to evaluation mode
    loss_function = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            y = y.float()

            outputs = model(X)
            loss = loss_function(outputs, y)

            val_loss.append(loss.item())

            preds = (torch.sigmoid(outputs) > 0.5).long()
            val_correct += torch.sum(preds == y).item()
            total_samples += y.size(0)

    avg_loss = np.mean(val_loss)
    accuracy = val_correct / total_samples

    return avg_loss, accuracy


def evaluate(model, dataloader, device):
    '''
    Evaluate model performance (return true labels + predicted probabilities).
    '''
    outputs = []
    true_labels = []

    model.eval()

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            probs = torch.sigmoid(logits)

            outputs += probs.cpu().numpy().tolist()
            true_labels += y.cpu().numpy().tolist()

    return np.array(true_labels), np.array(outputs)


def train_validate(model, train_loader, val_loader, optimizer, device, epoch_max=100):
    '''
    Perform training and validation across multiple epochs.
    '''
    val_loss_curve = []
    val_acc_curve = []
    train_loss_curve = []
    train_acc_curve = []

    early_stopping = EarlyStopping()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.1)
    #scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)


    tqdm_progress = tqdm(range(epoch_max), desc="Training Progress")

    for epoch in tqdm_progress:
        # Train for one epoch
        train_loss, train_acc = train(model, train_loader, optimizer, device)

        # Validate for one epoch
        val_loss, val_acc = validate(model, val_loader, device)

        # Scheduler step based on validation loss
        scheduler.step(val_loss)

        # Early stopping check
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

        # Track losses and accuracies
        train_loss_curve.append(train_loss)
        train_acc_curve.append(train_acc)
        val_loss_curve.append(val_loss)
        val_acc_curve.append(val_acc)

        tqdm_progress.set_postfix({
            'Train Loss': f'{train_loss:.4f}',
            'Train Acc': f'{train_acc:.4f}',
            'Val Loss': f'{val_loss:.4f}',
            'Val Acc': f'{val_acc:.4f}'
        })

    return {
        "train_loss_curve": train_loss_curve,
        "train_acc_curve": train_acc_curve,
        "val_loss_curve": val_loss_curve,
        "val_acc_curve": val_acc_curve
    }


### Training of model and performance

# **50/50 2k/2k dataset classifiers**

## **ChemBERTa LightGBM model**

### Dataset split

Data Loading

In [ ]:
import os
#path to the file
file_name_dubl = "A_blissind_abxinfo_DropArray.xlsx"
file_path = os.path.join(project_folder, file_name_dubl)

#install openpyxl
!pip install openpyxl

#load the Excel file
df = pd.read_excel(file_path)

# returns a NumPy array of the distinct antibiotic names
unique_abx = df["abx_name"].unique()

# Map abx SMILES
df["abx_SMILES"] = df["abx_name"].map(abx_smiles_map)


Data Split

In [ ]:

#label column
conditions = [
    df["bliss_med"] >= 0.30  # synergistic
]
choices = [1]

# Create synergy_label column with 1 for >=0.30 and NaN for the rest
df["synergy_label"] = np.select(conditions, choices, default=np.nan)

# Filter the rows with synergy label 1
synergistic_df = df[df["synergy_label"] == 1].copy()

# Calculate the number of 1 labels
num_synergistic = len(synergistic_df)

# Pick the same number of random rows from the rest (non-synergistic)
non_synergistic_df = df[df["synergy_label"].isna()].sample(n=num_synergistic, random_state=42).copy()

# Assign label 0 to the selected random rows
non_synergistic_df["synergy_label"] = 0

# Combine the two dataframes to achieve a 50/50 balance
balanced_df = pd.concat([synergistic_df, non_synergistic_df]).reset_index(drop=True)



# save the cleaned-up dataframe to disk
# balanced_df.to_csv("balanced_synergy_labeled.csv", index=False)


In [ ]:

full_df = pd.DataFrame(X)          # no copy of X
full_df["synergy_label"] = y.values       # just a view, no np.hstack needed


In [ ]:

# get embeddings
def get_smiles_embedding(smiles: str, model_name: str = "ChemBERTa-77M-MTR"):
    tok, lm = load_model(model_name)
    vec = smiles_to_vec(smiles, tok, lm)
    return vec

# Apply the embedding extraction to each row using tqdm for progress tracking
full_df["abx_embedding"] = [
    get_smiles_embedding(str(smiles)) for smiles in tqdm(full_df["abx_SMILES"])
]

full_df["cp_embedding"] = [
    get_smiles_embedding(str(smiles)) for smiles in tqdm(full_df["cp_SMILES"])
]

# Display the updated DataFrame
full_df.head()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


X = pd.DataFrame(
    full_df.apply(
        lambda row: np.concatenate([row["abx_embedding"], row["cp_embedding"]]), axis=1
    ).tolist()
)

# Assigning column names for better understanding
X.columns = [f"feature_{i}" for i in range(X.shape[1])]

# Adding the original columns (abx_conc and stain) if you want them in the feature set
X["abx_conc"] = full_df["abx_conc"].values
X['Ea_med'] = full_df['Ea_med'].values
# X['Ec_med'] = df_wo_emb['Ec_med'].values
X["strain_name_Ab17978"] = full_df["strain_name_Ab17978"].values
X["strain_name_AbLac4"] = full_df["strain_name_AbLac4"].values
X["strain_name_Kp0087"] = full_df["strain_name_Kp0087"].values
X["strain_name_Kp43816"] =  full_df["strain_name_Kp43816"].values
X["strain_name_PAO1"] = full_df["strain_name_PAO1"].values
X["strain_name_Pa0095"] = full_df["strain_name_Pa0095"].values


# Example target variable (replace with your actual target)
y = full_df['synergy_label']  # Replace this with your actual target column



# Displaying the resulting X DataFrame
print(X.head())



### LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
from sklearn.model_selection import KFold


# Create KFold object
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Store metrics for averaging
aucs = []
accuracies = []
conf_matrices = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    print(f"\n--- Fold {fold} ---")

    X_train, X_test = X.iloc[train_idx].astype(np.float32), X.iloc[test_idx].astype(np.float32)
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    clf = lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )

    try:
        clf.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, first_metric_only=True),
                lgb.log_evaluation(period=50),
            ],
        )
    except TypeError:
        clf.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="auc",
            early_stopping_rounds=50,
            verbose=False,
        )

    proba = clf.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, proba)
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred).ravel()  # TN, FP, FN, TP

    print(f"Best iteration : {clf.best_iteration_}")
    print(f"AUROC          : {auc:.4f}")
    print(f"Accuracy       : {acc:.4f}")
    print(f"Confusion matrix (TN, FP, FN, TP): {cm}")

    aucs.append(auc)
    accuracies.append(acc)
    conf_matrices.append(cm)

# Average metrics
avg_auc = np.mean(aucs)
avg_acc = np.mean(accuracies)
sum_cm = np.sum(conf_matrices, axis=0)

print("\n=== Cross-Validation Summary ===")
print(f"Average AUROC    : {avg_auc:.4f}")
print(f"Average Accuracy : {avg_acc:.4f}")
print(f"Total Confusion Matrix (sum over folds): {sum_cm}")


In [ ]:

# PLOTS for LightGBM model performance

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay
import numpy as np

#1) ROC curve
fpr, tpr, _ = roc_curve(y_test, proba)
roc_auc      = auc(fpr, tpr)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, lw=2, label=f"AUROC = {roc_auc:0.3f}")
plt.plot([0, 1], [0, 1], ls="--", lw=1)
plt.xlabel("False‑Positive Rate")
plt.ylabel("True‑Positive Rate")
plt.title("ROC curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# 2) Confusion‑matrix heatmap
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    cmap="Blues",
    colorbar=False,
)
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

# 3) Feature‑importance plot
# LightGBM returns importance values for every feature column (same order as X)
importances = clf.feature_importances_
# Grab the top‑N to keep the plot readable
N = 20
idx_top     = np.argsort(importances)[-N:]
plt.figure(figsize=(6, 4 + 0.25 * N))
plt.barh(range(N), importances[idx_top], align="center")
plt.yticks(range(N), [f"feat_{i}" for i in idx_top])
plt.xlabel("Gain‑based importance")
plt.title(f"Top {N} features")
plt.tight_layout()
plt.show()
